# [실습] PDF 기반 질의응답 시스템 구축

## API 키 설정 

In [1]:
# Load environment variables in a file called .env
# API 키 설정 
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

# Check the key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

API key found and looks good so far!
Google API Key exists and begins AI


없으면 설치

In [4]:
! uv pip install pymupdf

Using Python 3.12.12 environment at: C:\Users\jino7\Desktop\llm_engineering-main\.venv
Resolved 1 package in 275ms
 Downloaded pymupdf
Prepared 1 package in 1.19s
Installed 1 package in 52ms
 + pymupdf==1.27.2.3


## 1. PDF 문서 로드 및 전처리

In [24]:
from langchain_community.document_loaders import PyMuPDFLoader
import re

# ── 1. PDF 로드 ──────────────────────────────────────────────
# PyMuPDFLoader: PDF를 페이지 단위로 읽어 Document 객체 리스트로 반환
# 각 Document는 page_content(텍스트)와 metadata(페이지 번호 등)를 포함
pdf_path = 'test.pdf'  # 분석할 PDF 파일 경로 (필요 시 변경)
loader = PyMuPDFLoader(pdf_path)
pages = loader.load()  # 전체 페이지를 리스트로 불러옴

# ── 2. 페이지 텍스트 전처리 ───────────────────────────────────
# PDF에서 추출한 텍스트는 불필요한 공백·특수문자가 많아 정제가 필요
for page in pages:
    page.page_content = re.sub(
        r'[^\w\s]',   # 정규식: 영문자·숫자·한글·공백을 제외한 모든 특수문자 대상
        '',           # 매칭된 특수문자를 빈 문자열로 교체 (= 제거)
        page.page_content
            .replace('\xa0', '')   # \xa0 : HTML 엔티티 &nbsp; (줄바꿈 없는 공백) 제거
            .replace('\n\n', '\n') # 연속된 빈 줄을 단일 줄바꿈으로 압축
    )

# ── 3. 특정 페이지 확인 ───────────────────────────────────────
# pages는 0-index 리스트이므로 pages[3]은 PDF의 4번째 페이지에 해당
print(pages[3])  # 전처리 결과 확인용 출력 (인덱스 변경으로 원하는 페이지 확인 가능)


page_content='LGFA PLAYING Rules 2022 
2 
 
 
경기장 
1 
여자 게일식 축구는 U14 경기부터 GAA 풀사이즈 
경기장에서 진행됩니다 U13 이하 등급의 경우 경기장 
크기가 축소될 수 있습니다 
2 
15 인제 미만의 경기일 경우 조직 위원회는 경기장의 크기 
득점 공간 경기 시간을 단축할 수 있습니다 
 
선수 
1 
임신 가능성이 있거나 뇌진탕 등을 앓고 있는 선수는 여자 
게일식 축구를 할 수 없습니다 만약 출전하면 스스로 
전적인 위험을 감수해야 하며 여자 게일식 축구 협회는 
그에 따른 결과에 어떤 책임도 지지 않습니다 
 
선수 복장 
1 
여자 게일식 축구의 복장은 저지 반바지 양말 
부츠입니다 선수는 여자 게일식 축구를 하는 동안 
쥬얼리 귀걸이 머리핀 또는 부상을 유발할 수 있는 기타 
아이템을 착용할 수 없습니다 인조 표면에서 진행되는 
경기의 경우 팀 내 색상이 통일되었다는 전제하에 
선수들은 다리 보호대를 착용할 수 있습니' metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2022-09-02T03:22:38+09:00', 'source': 'test.pdf', 'file_path': 'test.pdf', 'total_pages': 11, 'format': 'PDF 1.7', 'title': '', 'author': 'user', 'subject': '', 'keywords': '', 'moddate': '2022-09-02T03:22:38+09:00', 'trapped': '', 'modDate': "D:20220902032238+09'00'", 'creationDate': "D:20220902032238+09'00'", 'page': 3}


## 벡터 DB 처리

In [25]:
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os  # os.path.exists 사용을 위해 필요 (원본 코드에 누락된 import 추가)

# ── 1. 텍스트 청크 분할 ───────────────────────────────────────
# RecursiveCharacterTextSplitter: 문서를 일정 크기의 청크(조각)로 나눔
# - chunk_size    : 청크 1개의 최대 글자 수 (1000자)
# - chunk_overlap : 인접 청크 간 겹치는 글자 수 (200자)
#                   → 청크 경계에서 문맥이 끊기지 않도록 앞뒤 내용을 일부 공유
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(pages)  # 전처리된 pages를 청크 리스트로 변환

# ── 2. 임베딩 모델 초기화 ─────────────────────────────────────
# OpenAIEmbeddings: 텍스트를 고차원 벡터(숫자 배열)로 변환하는 모델
# - text-embedding-3-large: OpenAI의 최신 고성능 임베딩 모델
#   (벡터 차원 3072, 유사도 검색 정확도가 높음)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# ── 3. 벡터 DB 경로 설정 ─────────────────────────────────────
# 벡터 DB를 로컬 디스크에 저장할 폴더명 지정
# 동일한 이름으로 재실행 시 기존 데이터가 누적되지 않도록 관리
db_name = "OpenAI_Embeddings_DB"

# ── 4. 기존 벡터 DB 초기화 ───────────────────────────────────
# 같은 폴더가 이미 존재하면 이전 실행의 데이터가 남아 있을 수 있음
# → 컬렉션을 삭제하여 중복 저장 방지 후 새로 생성
if os.path.exists(db_name):
    Chroma(
        persist_directory=db_name,      # 기존 DB 폴더 경로
        embedding_function=embeddings   # 동일한 임베딩 모델 지정
    ).delete_collection()               # 컬렉션(저장된 벡터 전체) 삭제

# ── 5. 벡터 DB 생성 및 저장 ──────────────────────────────────
# 청크 텍스트를 임베딩 벡터로 변환한 뒤 Chroma DB에 저장
# - from_documents : 문서 리스트를 받아 임베딩 → 저장을 한 번에 처리
# - persist_directory 지정 시 메모리가 아닌 디스크에 영구 저장
#   → 다음 실행 때 Chroma(persist_directory=db_name) 로 재사용 가능
vectorstore = Chroma.from_documents(
    documents=splits,          # 분할된 문서 청크 리스트
    embedding=embeddings,      # 텍스트 → 벡터 변환에 사용할 임베딩 모델
    persist_directory=db_name  # 벡터 DB를 저장할 로컬 폴더 경로
)

# ── 6. 저장 결과 확인 ─────────────────────────────────────────
# _collection.count(): Chroma 내부 컬렉션에 저장된 벡터(문서 청크) 수 반환
# split 수와 일치하면 모든 청크가 정상적으로 저장된 것
print(f"벡터 DB 생성 완료: {vectorstore._collection.count()}개의 문서 저장됨")
print(f"벡터 DB 생성 완료: {vectorstore._collection.count()}개의 문서 저장됨")


벡터 DB 생성 완료: 10개의 문서 저장됨
벡터 DB 생성 완료: 10개의 문서 저장됨


In [26]:
# ── 1. Retriever 생성 ─────────────────────────────────────────
# as_retriever(): 벡터 스토어를 검색기(Retriever) 객체로 변환
# Retriever는 질의(query)를 받아 관련 문서를 반환하는 인터페이스
# 기본 설정: 코사인 유사도 기반으로 가장 유사한 문서 4개 반환
# 옵션 예시:
#   as_retriever(search_type="mmr")                         → 다양성 고려 검색 (MMR 알고리즘)
#   as_retriever(search_kwargs={"k": 10})                   → 반환 문서 수를 10개로 변경
#   as_retriever(search_type="similarity_score_threshold",
#                search_kwargs={"score_threshold": 0.8})    → 유사도 0.8 이상인 문서만 반환
retriever = vectorstore.as_retriever()

# ── 2. 관련 문서 검색 ─────────────────────────────────────────
# invoke(): 질의 문자열을 입력받아 벡터 유사도 검색 수행
# 질의를 임베딩 벡터로 변환 → DB 내 청크 벡터들과 유사도 비교 → 상위 k개 반환
documents = retriever.invoke("축구 규칙을 설명하라")

# ── 3. 검색 결과 출력 ─────────────────────────────────────────
# 반환된 Document 객체 리스트를 순회하며 내용 확인
# document.page_content : 청크의 실제 텍스트 내용
# document.metadata     : 출처 파일명, 페이지 번호 등 부가 정보
for document in documents:
    print(f"문서 내용: {document.page_content}")

문서 내용: LGFA PLAYING Rules 2022 
1 
 
 
 
 
 
 
 
 
 
소개 
본 소책자는 여자 게일식 축구의 경기 규칙을 간략하게 
설명하고 있습니다 전체 경기 규칙은 공식 가이드에서 
확인하실 수 있습니다 
 
이 책자의 목표는 모든 선수와 관계자들이 경기 규칙에 
대해 읽고 이해하도록 하는 것입니다 교사와 코치의 
경우 반드시 이 책자를 소지하고 있는 것이 중요합니다 
그래야만 경기에 대한 이해력을 높일 수 있고 
관계자들의 결정을 이의 없이 받아들일 수 있을 
것입니다 
 
본 소책자는 협회와 마찬가지로 게일식 축구 경기에 대한 
이해 촉진을 증진하고자 합니다
문서 내용: LGFA PLAYING Rules 2022 
8 
 
 
경기 규칙 
1 
선수는 서 있는 자세에서 발가락 또는 한 손이나 두 
손으로 그라운드에 있는 공을 집어 들 수 있습니다 
 
2 
그라운드에 있는 선수는 자신으로부터 멀리 공을 차거나 
보낼 수 있지만 자신이 차지하기 위해 공을 가져올 수는 
없습니다 
 
3 
공을 잡으면 발로 차거나 주먹으로 치거나 손바닥으로 
때리거나 한 손이나 양손으로 한 번 튕기거나 발끝에서 
손으로 솔로할 수 있습니다 
 
4 
잡히지 않은 공은 한 손이나 두 손으로 연속해서 한 번 
이상 튕길 수 있습니다 
 
5 
이동이 완료될 때까지 원래 잡은 손이 공과의 접촉을 
유지하는 한 선수는 공을 한 손에서 다른 손으로 한 번 
옮길 수 있습니다 
 
6 
선수는 상대방이 공을 들어 올리거나 차지하려고 할 때 
공을 찰 수 없습니다 
 
7 
공을 4 걸음을 이동하여 5 걸음째 차는 데 필요한 
시간보다 오래 보유하지 않아야 합니다 
8 
선수는 공을 던질 수 없습니다 
9 
고의적인 신체 접촉은 없어야 합니다 
10 어깨싸움은 허용되지 않습니다
문서 내용: LGFA PLAYING Rules 2022 
3 
 
 
 
2 
여자 게일식 축구 경기 중 모든 선수는 자격을 갖춘 의사나 
치과의사가 따로 지시하지 않는 한 마우스피스를

# gpt 연동

In [27]:
# ── 1. 모듈 임포트 ────────────────────────────────────────────
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

# ── 2. 시스템 프롬프트 정의 ───────────────────────────────────
# 모델이 답변할 때 따라야 할 규칙과 형식을 지정
# {summaries}: 체인 실행 시 retriever가 검색한 문서 청크로 자동 채워짐
system_prompt = """'질의응답' 과정의 자료가 아래에 주어집니다. 질문에 대해 아래의 순서로 답변하세요. 한국어로 답변하세요.
1. 주어진 문서의 내용이 질문에 얼마나 적합한지 평가하세요.
2. 질문에 대한 답변을 출력하세요.
3. metadata page도 답하라
---
{summaries}
---
Answer:"""

# ── 3. 프롬프트 템플릿 구성 ───────────────────────────────────
# SystemMessagePromptTemplate : 모델의 역할·행동 방식을 정의하는 지침 메시지
# HumanMessagePromptTemplate  : 실제 사용자 질문을 전달하는 메시지
# {question}: 체인 실행 시 사용자 입력 query로 자동 채워짐
messages = [
    SystemMessagePromptTemplate.from_template(system_prompt),
    HumanMessagePromptTemplate.from_template("Question: {question}")
]

# 두 메시지 템플릿을 하나의 ChatPromptTemplate으로 결합
# → 최종적으로 [System 메시지, Human 메시지] 형태로 모델에 전달됨
prompt = ChatPromptTemplate.from_messages(messages)

# ── 4. LLM 초기화 ─────────────────────────────────────────────
# model_name  : 사용할 OpenAI 모델 (gpt-4.1-nano: 경량·저비용 모델)
# temperature : 출력의 창의성 조절 (0.0=일관된 답변 / 1.0=다양한 답변)
#               0.1로 설정 → 사실 기반의 안정적인 답변 유도
# max_tokens  : 모델이 생성할 최대 토큰 수 (4096 ≈ 약 3000 단어)
llm = ChatOpenAI(model_name="gpt-4.1-nano", temperature=0.1, max_tokens=4096)

# ── 5. RAG 체인 구성 (LCEL 파이프라인) ───────────────────────
# LangChain Expression Language(LCEL)의 | 연산자로 단계를 연결
#
# 실행 흐름:
#   query 입력
#     → {"summaries": retriever가 관련 문서 검색, "question": query 그대로 전달}
#     → prompt 템플릿에 summaries·question 삽입
#     → llm이 완성된 프롬프트로 답변 생성
#     → StrOutputParser가 AIMessage 객체에서 텍스트만 추출
#
# RunnablePassthrough: 입력값을 변환 없이 그대로 다음 단계로 전달
#   → query가 "question" 키 값으로 프롬프트에 전달됨
chain = (
    {"summaries": retriever, "question": RunnablePassthrough()}
    | prompt   # 검색 문서 + 질문을 프롬프트 형식으로 조합
    | llm      # 조합된 프롬프트를 LLM에 전달하여 답변 생성
    | StrOutputParser()  # LLM 응답(AIMessage)에서 순수 텍스트 문자열만 추출
)

# ── 6. 질의응답 실행 ──────────────────────────────────────────
# chain.invoke(query): 전체 RAG 파이프라인을 한 번에 실행
# 1) query → retriever → 관련 문서 검색
# 2) 검색 문서 + query → 프롬프트 완성
# 3) 완성된 프롬프트 → LLM 답변 생성
# 4) 생성된 답변 → 문자열로 파싱 후 반환
query = "이 문서에서 다루는 주요 주제는 무엇인가요?"
answer = chain.invoke(query)
print(answer)

1. 주어진 문서의 내용이 질문에 얼마나 적합한지 평가하세요.
- 문서는 여자 게일식 축구의 경기 규칙에 관한 내용을 다루고 있으며, 경기 시간, 선수의 보호장비, 출혈 및 두부 손상 규정 등 구체적인 규칙과 지침을 포함하고 있습니다. 따라서 질문에 매우 적합합니다.

2. 질문에 대한 답변을 출력하세요.
- 이 문서는 여자 게일식 축구의 경기 규칙과 관련된 내용을 다루고 있으며, 경기 시간, 선수 보호장비, 부상 시 조치, 교체 규정 등 경기 운영과 선수 안전에 관한 주요 규칙들을 설명하고 있습니다.

3. metadata page도 답하라
- 문서의 메타데이터는 다음과 같습니다:
  - 제목: 없음
  - 저자: user
  - 제작자: Microsoft® Word 2019
  - 생성일: 2022-09-02
  - 페이지 수: 11페이지
  - 파일명: test.pdf
  - 형식: PDF 1.7
